<a href="https://colab.research.google.com/github/Olga260584/Python/blob/main/%22HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import math

# Початкове та очікуване утримання
p1 = 0.19
p2 = 0.20

# Рівень значущості та потужність
alpha = 0.05
power = 0.80

# Розрахунок розміру ефекту
effect_size = proportion_effectsize(p1, p2)

# Розрахунок кількості користувачів на групу
analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,
    alternative='two-sided'
)

n_per_group = math.ceil(n_per_group)
total_users = n_per_group * 2

print("Користувачів на групу:", n_per_group)
print("Загальна кількість користувачів:", total_users)


Користувачів на групу: 24638
Загальна кількість користувачів: 49276


### Висновок до завдання 1

Для розрахунку необхідного розміру вибірки A/B-тесту використано бібліотеку `statsmodels`.

Початковий рівень утримання користувачів становить 19%, очікуваний - 20%. Мінімальний ефект, який потрібно виявити, дорівнює 1 відсотковому пункту.

Для розрахунку прийнято рівень значущості α = 0,05 та статистичну потужність 80%.

**Результати розрахунку:**
- Необхідна кількість користувачів в одній групі - 24 638.
- Загальна кількість користувачів для двох груп - 49 276.

Отже, для виявлення зазначеного ефекту необхідно залучити щонайменше **49 276 користувачів** за умови рівномірного розподілу між двома групами.

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [2]:

import pandas as pd

# Завантаження даних
df = pd.read_csv("cookie_cats.csv")

# Перегляд перших 5 рядків
df.head()


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [3]:

# Середнє утримання на 7-й день за групами
retention_by_version = df.groupby("version")["retention_7"].mean()

print("Утримання на 7-й день:")
print(retention_by_version)

# Переведення у відсотки
print("\nУтримання у відсотках:")
print(retention_by_version * 100)


Утримання на 7-й день:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

Утримання у відсотках:
version
gate_30    19.020134
gate_40    18.200004
Name: retention_7, dtype: float64


### Завдання 2. Аналіз утримання користувачів на 7-й день

За результатами аналізу даних отримано:

- **gate_30:** утримання на 7-й день - 19,02%.
- **gate_40:** утримання на 7-й день - 18,20%.

Різниця між групами становить приблизно 0,82 відсоткового пункту на користь `gate_30`.

**Формулювання гіпотез:**

**H₀:** p₃₀ = p₄₀ - утримання користувачів на 7-й день однакове для обох версій гри.

**H₁:** p₃₀ ≠ p₄₀ - утримання користувачів на 7-й день відрізняється між версіями гри.

**Попередній висновок:** за вибірковими даними версія `gate_30` демонструє краще утримання користувачів через 7 днів після встановлення гри. Для визначення статистичної значущості цієї різниці необхідно провести перевірку гіпотез.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [4]:

from statsmodels.stats.proportion import proportions_ztest

# Кількість гравців, які повернулися через 7 днів
successes = df.groupby("version")["retention_7"].sum()

# Загальна кількість гравців у кожній групі
nobs = df.groupby("version")["retention_7"].count()

# Z-тест для двох пропорцій
z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=nobs,
    alternative='two-sided'
)

print("z statistic:", z_stat)
print("p-value:", p_value)


z statistic: 3.164358912748191
p-value: 0.001554249975614329


In [5]:

from statsmodels.stats.proportion import proportion_confint

# Довірчі інтервали для обох груп
for version in ["gate_30", "gate_40"]:
    count = successes[version]
    n = nobs[version]

    ci_low, ci_high = proportion_confint(
        count=count,
        nobs=n,
        alpha=0.05,
        method="wilson"
    )

    group_name = "control" if version == "gate_30" else "treatment"

    print(f"Довірчий інтервал 95% для групи {group_name}: [{ci_low:.4f}, {ci_high:.4f}]")


Довірчий інтервал 95% для групи control: [0.1866, 0.1939]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1856]


### Висновок до завдання 3

Для порівняння утримання користувачів на 7-й день у двох версіях гри проведено двосторонній Z-тест для двох пропорцій на рівні значущості α = 0,05.

**Результати:**

- Z-статистика: 3,1644
- p-value: 0,001554
- 95% довірчий інтервал для групи control (`gate_30`): [0,1866; 0,1939]
- 95% довірчий інтервал для групи treatment (`gate_40`): [0,1785; 0,1856]

**1. Чи є статистично значуща різниця між версіями гри?**

Так. Оскільки p-value = 0,001554 < 0,05, нульову гіпотезу відхиляємо. Між версіями гри існує статистично значуща різниця в утриманні користувачів на 7-й день.

Версія `gate_30` демонструє вищий показник утримання (19,02%), ніж `gate_40` (18,20%).

**2. Чи перетинаються довірчі інтервали та про що це свідчить?**

Ні, 95% довірчі інтервали не перетинаються. Верхня межа інтервалу `gate_40` (18,56%) є нижчою за нижню межу інтервалу `gate_30` (18,66%).

Це додатково підтверджує наявність різниці між групами та узгоджується з результатом Z-тесту.

**Загальний висновок:** перенесення воріт із 30-го на 40-й рівень призвело до статистично значущого зниження утримання гравців на 7-й день. За цим показником доцільніше залишити ворота на 30-му рівні.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


### Завдання 4. Тест Хі-квадрат

**Нульова гіпотеза H₀:** версія гри та утримання користувачів на 7-й день є незалежними. Тобто версія гри не пов'язана з імовірністю повернення гравця через 7 днів.

**Альтернативна гіпотеза H₁:** між версією гри та утриманням користувачів на 7-й день існує статистично значущий зв'язок.

**Рівень значущості:** α = 0,05.

Для перевірки гіпотез використаємо критерій Хі-квадрат незалежності (χ²).

In [6]:

from scipy.stats import chi2_contingency
import pandas as pd

# Створюємо таблицю спряженості
contingency_table = pd.crosstab(
    df["version"],
    df["retention_7"]
)

print("Таблиця спряженості:")
print(contingency_table)

# Проводимо тест Хі-квадрат
chi2, p_value_chi2, dof, expected = chi2_contingency(
    contingency_table
)

print("\nХі-квадрат:", chi2)
print("p-value:", p_value_chi2)
print("Ступені свободи:", dof)


Таблиця спряженості:
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279

Хі-квадрат: 9.959086799559167
p-value: 0.0016005742679058301
Ступені свободи: 1


### Висновок до завдання 4

Для перевірки залежності між версією гри та утриманням користувачів на 7-й день проведено тест Хі-квадрат незалежності на рівні значущості α = 0,05.

**Отримані результати:**

- Статистика χ² = 9,9591
- p-value = 0,0016006
- Кількість ступенів свободи = 1

Оскільки p-value = 0,0016006 < 0,05, нульову гіпотезу H₀ відхиляємо.

Отже, між версією гри та поверненням користувачів через 7 днів існує статистично значущий зв'язок.

У групі `gate_30` утримання на 7-й день становить 19,02%, а у групі `gate_40` - 18,20%.

**Загальний висновок:** результати тесту Хі-квадрат підтверджують результати Z-тесту. Версія `gate_30` демонструє статистично значуще вище утримання гравців на 7-й день порівняно з `gate_40`.